# Instances with Holes

In [ ]:
from pathlib import Path
from zipfile import ZipFile

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from tspn_bnb2 import AnnotatedInstance
from tspn_bnb2.misc.paper_style import init_params

init_params()

## Load instances

In [ ]:
def load_instances(zip_path: str) -> dict[str, AnnotatedInstance]:
    instances = {}
    with ZipFile(zip_path, "r") as zf:
        for name in zf.namelist():
            if name.endswith(".json"):
                stem = Path(name).stem
                instances[stem] = AnnotatedInstance.model_validate_json(zf.open(name).read())
    return instances


def instance_type(inst: AnnotatedInstance) -> str:
    if "geo_information" in inst.meta:
        return "OSM"
    if inst.meta.get("source") == "random":
        return "random"
    return "tessellation"


def has_holes(inst: AnnotatedInstance) -> bool:
    return any(len(list(p.interiors)) > 0 for p in inst.polygons)


def count_holes(inst: AnnotatedInstance) -> int:
    return sum(len(list(p.interiors)) for p in inst.polygons)


socg = load_instances("../../instances/instances_socg.zip")
socg_simplified = load_instances("../../instances/instances_socg_simplified.zip")

print(f"instances_socg: {len(socg)}")
print(f"instances_socg_simplified: {len(socg_simplified)}")

## Count holes per instance

In [ ]:
rows = []
for name, inst in socg.items():
    rows.append(
        {
            "name": name,
            "n": len(inst.polygons),
            "instance_type": instance_type(inst),
            "dataset": "original",
            "has_holes": has_holes(inst),
            "num_holes": count_holes(inst),
        }
    )

for name, inst in socg_simplified.items():
    rows.append(
        {
            "name": name,
            "n": len(inst.polygons),
            "instance_type": instance_type(inst),
            "dataset": "simplified",
            "has_holes": has_holes(inst),
            "num_holes": count_holes(inst),
        }
    )

df = pd.DataFrame(rows)
df.head()

## Summary: instances with holes by instance type and dataset

In [ ]:
summary = (
    df.groupby(["instance_type", "dataset"])
    .agg(
        total=("has_holes", "count"),
        with_holes=("has_holes", "sum"),
        total_holes=("num_holes", "sum"),
    )
    .reset_index()
)
summary["with_holes"] = summary["with_holes"].astype(int)
summary["total_holes"] = summary["total_holes"].astype(int)
summary

## Plot

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(5.788, 2.5))

# Left: count of instances with holes
sns.countplot(
    data=df[df["has_holes"]],
    x="instance_type",
    hue="dataset",
    ax=axs[0],
)
axs[0].set_xlabel("Instance type")
axs[0].set_ylabel("Instances with holes")
axs[0].set_title("Instances containing holes")

# Right: total number of holes
sns.barplot(
    data=df.groupby(["instance_type", "dataset"])["num_holes"].sum().reset_index(),
    x="instance_type",
    y="num_holes",
    hue="dataset",
    ax=axs[1],
)
axs[1].set_xlabel("Instance type")
axs[1].set_ylabel("Total holes")
axs[1].set_title("Total hole count")

fig.tight_layout()